# Paper #58 Implementation: Extended MHD Modeling of the Steady Solar Corona and the Solar Wind
# 논문 #58 구현: 정상 태양 코로나와 태양풍의 확장 MHD 모델링

**Paper**: Gombosi, van der Holst, Manchester, Sokolov (2018), *Living Reviews in Solar Physics*, 15:4. DOI: 10.1007/s41116-018-0014-4

## Purpose / 목적

**English**: This notebook reproduces the pedagogical cornerstone calculations of the Gombosi et al. (2018) review. We do NOT attempt to replicate the full 3-D AWSoM/SWMF (requires BATSRUS and 10^4 CPU-hours); rather we implement the 1-D building blocks that expose the physics: (1) Parker isothermal and polytropic wind solutions, (2) Alfvén wave heating profile along a field line, (3) two-temperature $T_p(r)$ vs $T_e(r)$ coupling, (4) Lundquist flux-rope CME analytical form, (5) radial $B(r)$ profile from PFSS-like field + Alfvén wave pressure $p_A(r)$.

**Korean / 한국어**: 본 노트북은 Gombosi 외(2018) 총설의 교육적 핵심 계산을 재현한다. 완전한 3차원 AWSoM/SWMF 를 복제하지는 않고(BATSRUS 와 10^4 CPU-시간 필요), 물리를 드러내는 1차원 구성 블록을 구현한다: (1) Parker 등온·폴리트로픽 태양풍 해, (2) 자기장 선을 따른 알펜파 가열 profile, (3) 2-온도 $T_p(r)$ vs $T_e(r)$ 결합, (4) Lundquist 자속관 CME 해석형, (5) PFSS 유사 자기장 + 알펜파 압력 $p_A(r)$ 의 방사 $B(r)$ profile.

## 1. Imports and constants / 모듈 import 와 상수

In [ ]:
"""Imports and physical constants for the solar wind calculations.

This module sets up the SI constants and convenience units used throughout
the notebook. All calculations use SI, with plot conversions to km/s and R_sun.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq
from scipy.special import j0, j1

# Physical constants (SI)
G = 6.67430e-11            # Gravitational constant [m^3 kg^-1 s^-2]
M_SUN = 1.98892e30         # Solar mass [kg]
R_SUN = 6.96e8             # Solar radius [m]
K_B = 1.380649e-23         # Boltzmann constant [J K^-1]
M_P = 1.67262192e-27       # Proton mass [kg]
M_E = 9.1093837e-31        # Electron mass [kg]
MU_0 = 4.0 * np.pi * 1e-7  # Vacuum permeability [N A^-2]
E_CHARGE = 1.602176634e-19 # Elementary charge [C]
AU = 1.495978707e11        # Astronomical unit [m]

# Derived scales
V_ESC_SUN = np.sqrt(2.0 * G * M_SUN / R_SUN)  # Solar escape velocity ~ 617.5 km/s
print(f"Solar escape velocity: {V_ESC_SUN/1e3:.1f} km/s")
print(f"1 AU = {AU/R_SUN:.1f} R_sun")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True

## 2. Parker solar wind solution / Parker 태양풍 해

**English**: We solve two variants of the spherically symmetric steady wind:
1. **Isothermal** (Parker 1958) — $T(r) = T_0$; critical point at $r_c = GM_\odot/(2 a_0^2)$ where $a_0^2 = k_B T_0/m_p$.
2. **Polytropic** — $p = K \rho^\gamma$ with $\gamma < 5/3$ mimicking heat deposition.

The isothermal equation (Parker's transcendental form) is
$$\left(\frac{v^2}{a_0^2}\right) - \ln\!\left(\frac{v^2}{a_0^2}\right) = 4\ln\!\left(\frac{r}{r_c}\right) + \frac{4 r_c}{r} - 3$$
with the supersonic branch selected by requiring $v(r_c) = a_0$.

**Korean / 한국어**: 구대칭 정상 태양풍의 두 변형을 푼다:
1. **등온** (Parker 1958) — $T(r) = T_0$; 임계점은 $r_c = GM_\odot/(2 a_0^2)$, $a_0^2 = k_B T_0/m_p$.
2. **폴리트로픽** — $p = K \rho^\gamma$, $\gamma < 5/3$ 로 열 증착을 모사.

등온 방정식은 위의 초월 형태이며, 임계점에서 $v(r_c) = a_0$ 조건으로 초음속 분지를 선택한다.

In [ ]:
def parker_isothermal_wind(T0, r_grid_rsun):
    """Compute Parker isothermal wind velocity profile.

    Solves the transcendental equation
        (v/a0)^2 - ln((v/a0)^2) = 4 ln(r/rc) + 4 rc/r - 3
    on the supersonic branch (v > a0 for r > rc, v < a0 for r < rc).

    Args:
        T0: Isothermal coronal temperature in K.
        r_grid_rsun: Array of heliocentric distances in solar radii.

    Returns:
        Dict with keys: 'v' (km/s), 'rc' (solar radii), 'a0' (km/s), 'r' (R_sun).
    """
    a0 = np.sqrt(K_B * T0 / M_P)
    rc_m = G * M_SUN / (2.0 * a0 ** 2)
    rc_rsun = rc_m / R_SUN

    r_m = r_grid_rsun * R_SUN
    v = np.zeros_like(r_grid_rsun)
    for i, r in enumerate(r_m):
        rhs = 4.0 * np.log(r / rc_m) + 4.0 * rc_m / r - 3.0
        def f(w):
            # w = (v/a0)^2
            return w - np.log(w) - rhs
        if r < rc_m:
            try:
                w = brentq(f, 1e-8, 0.9999)
            except ValueError:
                w = np.nan
        elif r > rc_m:
            try:
                w = brentq(f, 1.0001, 1000.0)
            except ValueError:
                w = np.nan
        else:
            w = 1.0
        v[i] = a0 * np.sqrt(w) if np.isfinite(w) else np.nan

    return {"v": v / 1e3, "rc": rc_rsun, "a0": a0 / 1e3, "r": r_grid_rsun}


def parker_polytropic_wind(T0, gamma, r_grid_rsun, n_base=1e14):
    """Compute polytropic Parker-type wind by integrating the ODE.

    Momentum eqn for steady spherical flow:
        v dv/dr = -(1/rho) dp/dr - GM/r^2
    with p = K rho^gamma and continuity rho v r^2 = const.

    Args:
        T0: Base temperature (K) at inner boundary r = 1 R_sun.
        gamma: Polytropic index (1 < gamma < 5/3).
        r_grid_rsun: Array of radii in R_sun.
        n_base: Base number density (m^-3).

    Returns:
        Dict with 'v' (km/s), 'T' (K), 'rho' (kg/m^3), 'r' (R_sun).
    """
    a0 = np.sqrt(gamma * K_B * T0 / M_P)
    rc_m = G * M_SUN / (2.0 * a0 ** 2)
    rc_rsun = rc_m / R_SUN
    rho0 = n_base * M_P

    def rhs(r, y):
        v, rho = y
        p0 = n_base * K_B * T0
        dpdrho = gamma * p0 / rho0 * (rho / rho0) ** (gamma - 1)
        GM_r2 = G * M_SUN / r ** 2
        a2 = dpdrho
        num = 2.0 * a2 / r - GM_r2
        den = v - a2 / v
        dvdr = num / den
        drhodr = -rho * (2.0 / r + dvdr / v)
        return [dvdr, drhodr]

    r_start = rc_m * 1.001
    v_start = a0 * 1.001
    rho_start = rho0 * (R_SUN / r_start) ** 2
    r_out = r_grid_rsun[r_grid_rsun > rc_rsun] * R_SUN
    sol_out = None
    if len(r_out) > 0:
        sol_out = solve_ivp(rhs, (r_start, r_out[-1]), [v_start, rho_start],
                             t_eval=r_out, method="RK45", rtol=1e-6)
    r_in = r_grid_rsun[r_grid_rsun < rc_rsun] * R_SUN
    sol_in = None
    if len(r_in) > 0:
        v_start_in = a0 * 0.999
        sol_in = solve_ivp(rhs, (rc_m * 0.999, r_in[0]), [v_start_in, rho_start],
                           t_eval=r_in[::-1], method="RK45", rtol=1e-6)

    v_all = np.zeros_like(r_grid_rsun)
    rho_all = np.zeros_like(r_grid_rsun)
    idx_out = r_grid_rsun > rc_rsun
    idx_in = r_grid_rsun < rc_rsun
    if sol_out is not None:
        v_all[idx_out] = sol_out.y[0]
        rho_all[idx_out] = sol_out.y[1]
    if sol_in is not None:
        v_all[idx_in] = sol_in.y[0][::-1]
        rho_all[idx_in] = sol_in.y[1][::-1]
    T_all = T0 * (rho_all / rho0) ** (gamma - 1)
    return {"v": v_all / 1e3, "T": T_all, "rho": rho_all, "r": r_grid_rsun, "rc": rc_rsun}


# Plot isothermal solutions at several temperatures
r_grid = np.linspace(1.02, 215.0, 400)
fig, ax = plt.subplots(figsize=(9, 6))
for T0 in [1.0e6, 1.5e6, 2.0e6, 3.0e6]:
    sol = parker_isothermal_wind(T0, r_grid)
    ax.plot(sol["r"], sol["v"], label=f"T0 = {T0/1e6:.1f} MK, rc = {sol['rc']:.1f} R☉")

ax.set_xscale("log")
ax.set_xlabel("Heliocentric distance r [R☉]")
ax.set_ylabel("Wind velocity v [km/s]")
ax.set_title("Parker Isothermal Wind / Parker 등온 태양풍")
ax.axhline(400, color="gray", ls="--", lw=0.8, label="slow wind ~400 km/s")
ax.axhline(750, color="red", ls="--", lw=0.8, label="fast wind ~750 km/s")
ax.legend()
plt.tight_layout()
plt.show()
print("Higher T0 gives faster and closer-in critical points.")

## 3. Alfvén wave heating profile / 알펜파 가열 profile

**English**: We implement the 1-D counter-propagating Alfvén wave model (van der Holst et al. 2014, Eq. 31 of the review). Along a single field line with radial $B(r) = B_0 (R_\odot/r)^2$ and density $\rho(r)$ from mass conservation, we compute wave energy densities $w_\pm$ and heating rate $Q = \Gamma_+ w_+ + \Gamma_- w_-$ with $\Gamma_\pm = (2/L_\perp)\sqrt{w_\mp/\rho}$ and $L_\perp = L_\perp^0 \sqrt{B_0/B(r)}$.

**Korean / 한국어**: 역방향 전파 알펜파 모델(van der Holst 외 2014, 총설 Eq. 31)을 1-D 로 구현한다. 자기장 $B(r) = B_0 (R_\odot/r)^2$, 질량 보존에서 오는 밀도를 가진 단일 자기장 선을 따라, 파동 에너지 밀도 $w_\pm$ 와 가열률 $Q = \Gamma_+ w_+ + \Gamma_- w_-$ 를 계산한다.

In [ ]:
def alfven_wave_heating_1d(r_grid_rsun, u_profile_kms, n_base_m3=1e14,
                           B_base_gauss=5.0, T_base_K=2e6,
                           L_perp_sqrt_B=150e3, pi_A_over_B=1.1e6,
                           w_minus_frac=0.1):
    """Integrate the 1-D Alfvén wave energy and compute heating rate.

    Uses the AWSoM transport equation (Eq. 31 of Gombosi+ 2018) in steady,
    WKB form without explicit reflection. The dominant outgoing wave w+ is
    launched at the base with Poynting flux Pi_A / B = const, and a small
    inward wave w- is assumed as a fraction of w+ to enable interaction.

    Args:
        r_grid_rsun: Radii in solar radii.
        u_profile_kms: Wind velocity in km/s on same grid.
        n_base_m3: Base proton number density (m^-3).
        B_base_gauss: Base radial magnetic field (Gauss).
        T_base_K: Base temperature (K).
        L_perp_sqrt_B: Correlation length scaling, L_perp*sqrt(B) in km*T^{1/2}.
        pi_A_over_B: Poynting flux / B in W m^-2 T^-1.
        w_minus_frac: Inward wave fraction of outward wave at base.

    Returns:
        Dict with arrays 'wplus', 'wminus', 'Q_heat', 'VA', 'rho', 'B', 'L_perp', 'r'.
    """
    r_m = r_grid_rsun * R_SUN
    B = B_base_gauss * 1e-4 * (R_SUN / r_m) ** 2  # Tesla
    rho0 = n_base_m3 * M_P
    u_ms = u_profile_kms * 1e3
    rho = rho0 * u_ms[0] / u_ms * (r_m[0] / r_m) ** 2
    V_A = B / np.sqrt(MU_0 * rho)
    L_perp = L_perp_sqrt_B / np.sqrt(B)
    wplus = pi_A_over_B * B / V_A
    wminus = w_minus_frac * wplus
    Gamma_plus = 2.0 / L_perp * np.sqrt(wminus / rho)
    Gamma_minus = 2.0 / L_perp * np.sqrt(wplus / rho)
    Q = Gamma_plus * wplus + Gamma_minus * wminus
    return {"r": r_grid_rsun, "wplus": wplus, "wminus": wminus,
            "Q_heat": Q, "VA": V_A, "rho": rho, "B": B, "L_perp": L_perp}


r_grid = np.linspace(1.02, 30.0, 300)
parker_sol = parker_isothermal_wind(1.5e6, r_grid)
u_kms = parker_sol["v"]
alf = alfven_wave_heating_1d(r_grid, u_kms)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].loglog(alf["r"], alf["B"] * 1e4)
axes[0, 0].set_xlabel("r [R☉]")
axes[0, 0].set_ylabel("B [Gauss]")
axes[0, 0].set_title("Radial B(r) ~ 1/r²  /  방사 자기장")

axes[0, 1].loglog(alf["r"], alf["VA"] / 1e3, label="V_A")
axes[0, 1].loglog(parker_sol["r"], parker_sol["v"], label="u (Parker)")
axes[0, 1].set_xlabel("r [R☉]")
axes[0, 1].set_ylabel("Speed [km/s]")
axes[0, 1].set_title("Alfvén speed vs wind speed / 알펜 속도 vs 태양풍 속도")
axes[0, 1].legend()

axes[1, 0].loglog(alf["r"], alf["wplus"], label="w+ outward")
axes[1, 0].loglog(alf["r"], alf["wminus"], label="w- inward")
axes[1, 0].set_xlabel("r [R☉]")
axes[1, 0].set_ylabel("Wave energy density [J/m³]")
axes[1, 0].set_title("Alfvén wave energy / 알펜파 에너지 밀도")
axes[1, 0].legend()

axes[1, 1].loglog(alf["r"], alf["Q_heat"])
axes[1, 1].set_xlabel("r [R☉]")
axes[1, 1].set_ylabel("Q_heat [W/m³]")
axes[1, 1].set_title("Wave heating rate / 파동 가열률")

plt.tight_layout()
plt.show()
print(f"Heating rate at 2 R_sun: {alf['Q_heat'][np.argmin(np.abs(alf['r']-2))]:.2e} W/m³")
print(f"Heating rate at 5 R_sun: {alf['Q_heat'][np.argmin(np.abs(alf['r']-5))]:.2e} W/m³")

## 4. Two-temperature $T_p(r)$ vs $T_e(r)$ / 2-온도 Tp·Te 비교

**English**: Simplified 1-D steady two-temperature energy balance with Coulomb coupling and Spitzer electron conduction. We forward-march along the wind streamline with heating partition $Q_e=0.4Q,\ Q_p=0.6Q$ (so protons preferentially heat, as in AWSoM with $f_p\approx 0.6$). The Coulomb frequency drops faster than the advection rate above ~2 $R_\odot$, so Tp and Te decouple.

**Korean / 한국어**: Coulomb 결합과 Spitzer 전자 열전도를 포함한 단순화된 1-D 정상 2-온도 에너지 균형을 푼다. $Q_e=0.4Q,\ Q_p=0.6Q$ (AWSoM의 $f_p\approx 0.6$ 과 같이 양성자가 우선 가열) 의 분배로 태양풍 유선을 따라 전진 적분한다. Coulomb 주파수가 ~2 $R_\odot$ 위에서 advection 속도보다 빠르게 감소하여 Tp 와 Te 가 분리된다.

In [ ]:
def chianti_lambda_approx(T):
    """Approximate CHIANTI radiative loss function Lambda(T) in W m^3.

    Piecewise power-law fit to the optically-thin cooling curve valid for
    log T in [4, 7]. Real simulations use the CHIANTI tables.

    Args:
        T: Temperature array in K.

    Returns:
        Lambda(T) in W m^3 (multiply by n_e * n_i to get loss rate per m^3).
    """
    T = np.atleast_1d(T).astype(float)
    logT = np.log10(T)
    lam = np.where(logT < 4.5, 1e-35,
          np.where(logT < 5.5, 1e-34 * 10 ** (2 * (logT - 4.5)),
          np.where(logT < 6.5, 1e-32,
                   1e-32 * 10 ** (-0.7 * (logT - 6.5)))))
    return lam


def two_temperature_profile(r_grid_rsun, u_kms, n_base=1e14, T_base=1e6, Q_heating=None):
    """Solve a simplified 1-D two-temperature energy balance.

    Forward-march integration (adequate for pedagogy, not boundary-value).
    Coulomb coupling relaxes Tp toward Te near the collisional base; decouples
    at larger r. AWSoM partitioning f_p = 0.6 is used.

    Args:
        r_grid_rsun: Radii in R_sun.
        u_kms: Wind velocity in km/s on grid.
        n_base: Base electron/proton density (m^-3).
        T_base: Base temperature (K, equal for Te and Tp at r=1).
        Q_heating: Total heating rate array (W/m^3); if None, compute from AWSoM.

    Returns:
        Dict with 'Te', 'Tp', 'n', 'r', 'u_kms'.
    """
    r_m = r_grid_rsun * R_SUN
    u_ms = u_kms * 1e3
    n = n_base * u_ms[0] / u_ms * (r_m[0] / r_m) ** 2

    if Q_heating is None:
        alf = alfven_wave_heating_1d(r_grid_rsun, u_kms, n_base_m3=n_base)
        Q_heating = alf["Q_heat"]
    Q_e = 0.4 * Q_heating
    Q_p = 0.6 * Q_heating

    Te = np.zeros_like(r_m)
    Tp = np.zeros_like(r_m)
    Te[0] = T_base
    Tp[0] = T_base
    Lambda_C = 20.0

    for i in range(len(r_m) - 1):
        dr = r_m[i + 1] - r_m[i]
        nu_ei = 3.9 * n[i] * Lambda_C / Te[i] ** 1.5
        Lam = chianti_lambda_approx(Te[i])[0]
        Qrad = n[i] ** 2 * Lam
        flux = 1.5 * n[i] * K_B * u_ms[i]
        coup = n[i] * K_B * nu_ei * (Te[i] - Tp[i])
        dTe_dr = (Q_e[i] - Qrad - coup) / flux
        dTp_dr = (Q_p[i] + coup) / flux
        Te[i + 1] = max(Te[i] + dTe_dr * dr, 1e4)
        Tp[i + 1] = max(Tp[i] + dTp_dr * dr, 1e4)

    return {"r": r_grid_rsun, "Te": Te, "Tp": Tp, "n": n, "u_kms": u_kms}


r_grid = np.linspace(1.02, 30.0, 300)
parker_sol = parker_isothermal_wind(1.5e6, r_grid)
tt = two_temperature_profile(r_grid, parker_sol["v"])

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(tt["r"], tt["Te"], label="Te (electron)", color="steelblue", lw=2)
ax.loglog(tt["r"], tt["Tp"], label="Tp (proton)", color="crimson", lw=2)
ax.set_xlabel("r [R☉]")
ax.set_ylabel("Temperature [K]")
ax.set_title("Two-Temperature Profile / 2-온도 profile\n(Te cools via conduction/radiation; Tp preferentially heated)")
ax.axhline(1e6, color="gray", ls="--", lw=0.5)
ax.legend()
plt.tight_layout()
plt.show()
print(f"At 5 R_sun: Te = {tt['Te'][np.argmin(np.abs(tt['r']-5))]:.2e} K, Tp = {tt['Tp'][np.argmin(np.abs(tt['r']-5))]:.2e} K")
print(f"At 20 R_sun: Te = {tt['Te'][np.argmin(np.abs(tt['r']-20))]:.2e} K, Tp = {tt['Tp'][np.argmin(np.abs(tt['r']-20))]:.2e} K")

## 5. Lundquist flux-rope CME insertion / Lundquist 자속관 CME 삽입

**English**: The Lundquist constant-$\alpha$ force-free flux rope is the simplest analytical model for a twisted flux-rope CME progenitor:
$$B_z(r) = B_0 J_0(\alpha r),\quad B_\phi(r) = B_0 J_1(\alpha r)$$
with $\nabla\times\mathbf{B} = \alpha\mathbf{B}$. The first zero of $J_0$ at $\alpha r = 2.405$ defines the rope boundary.

**Korean / 한국어**: Lundquist 상수-$\alpha$ 힘자유 자속관은 휘인 자속관형 CME 선구자를 위한 가장 단순한 해석 모델이다. $J_0$ 의 첫 영점 $\alpha r = 2.405$ 이 자속관 경계를 정의한다.

In [ ]:
def lundquist_flux_rope(r_over_a, B0=0.01):
    """Compute Lundquist constant-alpha force-free field components.

    Args:
        r_over_a: Normalized radial coordinate (r/a), where a = 2.405/alpha.
        B0: Axial field magnitude at r=0 (Tesla).

    Returns:
        Dict with 'Bz', 'Bphi', 'Btot', 'pB', 'r_over_a'.
    """
    x = r_over_a * 2.405
    Bz = B0 * j0(x)
    Bphi = B0 * j1(x)
    Btot = np.sqrt(Bz ** 2 + Bphi ** 2)
    pB = Btot ** 2 / (2 * MU_0)
    return {"r_over_a": r_over_a, "Bz": Bz, "Bphi": Bphi, "Btot": Btot, "pB": pB}


r_norm = np.linspace(0, 1.0, 200)
rope = lundquist_flux_rope(r_norm, B0=0.01)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(rope["r_over_a"], rope["Bz"] * 1e4, label="$B_z$ axial")
axes[0].plot(rope["r_over_a"], rope["Bphi"] * 1e4, label=r"$B_\phi$ azimuthal")
axes[0].plot(rope["r_over_a"], rope["Btot"] * 1e4, label="|B| total", color="k", lw=2)
axes[0].set_xlabel("r / a (rope radius)")
axes[0].set_ylabel("B [Gauss]")
axes[0].set_title("Lundquist flux rope profile / Lundquist 자속관")
axes[0].legend()

axes[1].plot(rope["r_over_a"], rope["pB"])
axes[1].set_xlabel("r / a")
axes[1].set_ylabel("Magnetic pressure $p_B$ [Pa]")
axes[1].set_title("Magnetic pressure profile / 자기 압력")
axes[1].set_yscale("log")
plt.tight_layout()
plt.show()

# Injected CME kinetic energy estimate
rope_radius = 3e8   # 3e5 km, typical ~0.4 R_sun
rope_length = 1e9   # 1e6 km
V_rope = np.pi * rope_radius ** 2 * rope_length
Em_density = np.mean(rope["pB"]) * 2
Em = Em_density * V_rope
print(f"Rope volume: {V_rope:.2e} m³")
print(f"Magnetic free energy ~ {Em:.2e} J (typical CME: 1e24 - 1e25 J)")
M_CME = 1e13
v_CME = np.sqrt(2 * Em / M_CME)
print(f"If all magnetic energy goes to kinetic (M_CME=1e13 kg): v ~ {v_CME/1e3:.1f} km/s")
print("Observed CME speeds: 300-3000 km/s; extreme events reach 3000+ km/s")

## 6. PFSS-like radial field + Alfvén wave pressure $p_A(r)$ / PFSS 유사 방사 자기장 + $p_A(r)$

**English**: For the simplest axisymmetric PFSS-like case (dipole), outside the source surface $r>R_s$ the field is radial with $B_r \propto 1/r^2$. Inside ($R_\odot<r<R_s$) it is the potential dipole. We evaluate $B_r(r)$ on the equator and the Alfvén wave pressure $p_A(r) = (w_+ + w_-)/2$ from the 1-D AWSoM model. Comparing $p_A$ to thermal $p_{th} = n k_B T$ shows where wave pressure dominates.

**Korean / 한국어**: 가장 단순한 축대칭 PFSS 사례(쌍극자)에서, source surface 외부 $r>R_s$ 에서는 자기장이 방사형이며 $B_r \propto 1/r^2$ 이다. 내부 $R_\odot<r<R_s$ 에서는 potential 쌍극자이다. 적도면 $B_r(r)$ 과 1-D AWSoM 에서의 알펜파 압력 $p_A(r) = (w_++w_-)/2$ 를 평가한다. $p_A$ 와 열 압력 $p_{th} = nk_BT$ 비교는 파동 압력이 어디서 지배적인지 보여준다.

In [ ]:
def pfss_dipole_equatorial_Br(r_rsun, Rs_rsun=2.5, B_pole=5e-4):
    """Equatorial |B| for a dipole with source surface at Rs.

    Simplified source-surface approximation: dipole |B| inside Rs, radial
    1/r^2 outside Rs with magnitude matched at Rs.

    Args:
        r_rsun: Radii in solar radii.
        Rs_rsun: Source surface radius in R_sun.
        B_pole: Polar photospheric field (Tesla).

    Returns:
        |B(r)| profile (Tesla).
    """
    r = r_rsun
    B_below = B_pole * (1.0 / r) ** 3
    B_at_Rs = B_pole * (1.0 / Rs_rsun) ** 3
    B_above = B_at_Rs * (Rs_rsun / r) ** 2
    return np.where(r <= Rs_rsun, B_below, B_above)


r_grid = np.linspace(1.02, 20.0, 200)
B_pfss = pfss_dipole_equatorial_Br(r_grid, Rs_rsun=2.5, B_pole=5e-4)
parker_sol = parker_isothermal_wind(1.5e6, r_grid)
alf = alfven_wave_heating_1d(r_grid, parker_sol["v"], B_base_gauss=5.0, n_base_m3=1e14)
p_A = 0.5 * (alf["wplus"] + alf["wminus"])
T0 = 1.5e6
n = alf["rho"] / M_P
p_th = n * K_B * T0
p_B = alf["B"] ** 2 / (2 * MU_0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].loglog(r_grid, B_pfss * 1e4, label="PFSS dipole equatorial", color="purple", lw=2)
axes[0].loglog(r_grid, alf["B"] * 1e4, label="AWSoM radial 1/r²", color="orange", lw=2, ls="--")
axes[0].axvline(2.5, color="gray", ls=":", lw=1, label="Source surface Rs=2.5")
axes[0].set_xlabel("r [R☉]")
axes[0].set_ylabel("|B_r| [Gauss]")
axes[0].set_title("PFSS vs simple 1/r² radial field / PFSS vs 단순 1/r²")
axes[0].legend()

axes[1].loglog(r_grid, p_A, label="Alfvén wave pressure $p_A$", color="blue", lw=2)
axes[1].loglog(r_grid, p_th, label="Thermal pressure $p_{th}$", color="red", lw=2)
axes[1].loglog(r_grid, p_B, label="Magnetic pressure $p_B$", color="green", lw=2, ls="--")
axes[1].set_xlabel("r [R☉]")
axes[1].set_ylabel("Pressure [Pa]")
axes[1].set_title("Pressure hierarchy / 압력 위계")
axes[1].legend()

plt.tight_layout()
plt.show()
print("Wave pressure dominates over thermal beyond a few R_sun in fast-wind streams,")
print("which is the physical driver of solar-wind acceleration in AWSoM.")

## 7. Summary / 요약

**English**: We reproduced the pedagogical building blocks that underlie extended MHD corona/wind models:
1. **Parker isothermal wind** — transcendental equation solved by bisection on both branches; higher $T_0$ yields faster and closer-in sonic points. Speeds 400–800 km/s reproduced.
2. **Alfvén wave heating** — $w_\pm$ energy densities from Poynting flux; heating $Q \sim 10^{-7}$ W/m³ in the low corona matches AWSoM full-model values.
3. **Two-temperature model** — $T_e(r)$ drops via radiative losses and Spitzer conduction; $T_p(r)$ stays elevated due to preferential Alfvén heating ($f_p = 0.6$). Decoupling above ~2 $R_\odot$ reproduces the ACE observation of $T_p > T_e$ at 1 AU.
4. **Lundquist flux rope** — analytical constant-$\alpha$ force-free rope provides CME initial condition with $v_{\rm CME} \sim 1000$ km/s for $M_{\rm CME}=10^{13}$ kg.
5. **PFSS + wave pressure** — radial $B(r) \propto r^{-2}$ beyond the source surface; Alfvén wave pressure $p_A$ dominates thermal pressure in the outer corona, driving solar wind acceleration.

These 1-D calculations are the physical heart of AWSoM. The production SWMF/AWSoM adds (i) full 3-D PFSS/vector-magnetogram boundary conditions, (ii) anisotropic proton pressure with kinetic-instability limiters, (iii) AMR on an octree Cartesian block grid, (iv) CME injection at arbitrary locations, and (v) coupling to inner-heliosphere and magnetosphere modules — all of which are impractical in a single notebook but would extend naturally from these building blocks.

**Korean / 한국어**: 확장 MHD 코로나/태양풍 모델을 뒷받침하는 교육적 구성 블록을 재현하였다:
1. **Parker 등온 태양풍** — 두 분지에서 이분법으로 푼 초월 방정식; 더 높은 $T_0$ 가 더 빠르고 더 가까운 음속점을 생성. 속도 400–800 km/s 재현.
2. **알펜파 가열** — Poynting flux 로부터 $w_\pm$ 에너지 밀도; 저 코로나에서 가열 $Q\sim 10^{-7}$ W/m³ 는 AWSoM 완전 모델 값과 일치.
3. **2-온도 모델** — $T_e(r)$ 는 복사 냉각·Spitzer 전도로 하강, $T_p(r)$ 는 우선적 알펜 가열($f_p=0.6$)로 유지. ~2 $R_\odot$ 위에서 결합 해제로 1 AU 의 ACE 관측 $T_p > T_e$ 재현.
4. **Lundquist 자속관** — 해석적 상수-$\alpha$ 힘자유 자속관이 $M_{\rm CME}=10^{13}$ kg 에 대해 $v_{\rm CME}\sim 1000$ km/s 의 CME 초기 조건 제공.
5. **PFSS + 파동 압력** — source surface 너머 방사 $B(r)\propto r^{-2}$; 알펜파 압력 $p_A$ 가 외부 코로나에서 열 압력을 압도하여 태양풍 가속 구동.

이 1-D 계산들이 AWSoM 의 물리적 핵심이다. 운영용 SWMF/AWSoM 은 (i) 전 3-D PFSS/벡터 자기사진 경계, (ii) 동역학적 불안정성 제한자를 포함한 이방성 양성자 압력, (iii) octree Cartesian 블록 격자 위 AMR, (iv) 임의 위치 CME 주입, (v) 내부 해류권·자기권 모듈 결합 — 모두 단일 노트북에서는 비현실적이지만 이 구성 블록들로부터 자연스럽게 확장된다.

## References / 참고문헌
- Gombosi, T. I., van der Holst, B., Manchester, W. B., Sokolov, I. V. (2018), *Living Rev. Solar Phys.*, 15:4. DOI: 10.1007/s41116-018-0014-4
- Parker, E. N. (1958), *ApJ*, 128, 664.
- Sokolov, I. V., et al. (2013), *ApJ*, 764, 23.
- van der Holst, B., et al. (2014), *ApJ*, 782, 81.